In [3]:
!pip install gymnasium

import gymnasium as gym
import numpy as np
from gymnasium import spaces
from gymnasium.envs.registration import register


class RecyclingRobotEnv(gym.Env):

    def __init__(self):
        super().__init__()

        # States: 0 = High battery, 1 = Low battery
        self.observation_space = spaces.Discrete(2)

        # Actions: 0 = Search, 1 = Wait, 2 = Recharge
        self.action_space = spaces.Discrete(3)

        # MDP Parameters (Sutton & Barto probabilities/rewards)
        self.alpha = 0.8
        self.beta = 0.5
        self.r_search = 2.0
        self.r_wait = 0.5

        self.state = 0  # Start at High battery

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        self.state = 0  # Always start at High
        return self.state, {}

    def step(self, action):
        reward = 0.0
        terminated = False
        truncated = False

        if self.state == 0:  # HIGH BATTERY

            if action == 0:  # Search
                if np.random.rand() < self.alpha:
                    self.state = 0
                else:
                    self.state = 1

                reward = self.r_search

            elif action == 1:  # Wait
                self.state = 0
                reward = self.r_wait

            elif action == 2:  # Recharge (No-op in High)
                self.state = 0
                reward = 0.0

        elif self.state == 1:  # LOW BATTERY

            if action == 0:  # Search
                if np.random.rand() < self.beta:
                    self.state = 1
                    reward = self.r_search
                else:  # Battery dies!
                    self.state = 0  # Rescued to High state
                    reward = -3.0
                    terminated = True

            elif action == 1:  # Wait
                self.state = 1
                reward = self.r_wait

            elif action == 2:  # Recharge
                self.state = 0
                reward = 0.0

        return self.state, reward, terminated, truncated, {}


# Register the custom environment
try:
    register(
        id="RecyclingRobot-v0",
        entry_point=RecyclingRobotEnv,
    )
except Exception:
    pass


# Run the simulation
env = None

try:
    env = gym.make("RecyclingRobot-v0")
    print("Successfully created RecyclingRobot-v0 environment.")

except Exception as e:
    print(f"Error creating environment: {e}")


if env:
    state, info = env.reset()

    print(f"\nAction Space: {env.action_space}")
    print(f"Observation Space: {env.observation_space}")

    def describe_state(state_data):
        state_names = {
            0: "High Battery",
            1: "Low Battery"
        }

        print(
            f"Current State: {state_data} "
            f"({state_names.get(state_data, 'Unknown')})"
        )

    print("\nInitial State:")
    describe_state(state)

    actions_map = {
        0: "Search",
        1: "Wait",
        2: "Recharge"
    }

    num_steps = 15

    print("\nSimulation for a few steps:")

    for step in range(num_steps):
        action = env.action_space.sample()

        next_state, reward, terminated, truncated, info = env.step(action)
        done = terminated or truncated

        print(f"\n--- Step {step + 1} ---")
        print(f"Action Chosen: {action} ({actions_map.get(action)})")

        describe_state(next_state)

        print(f"Reward Received: {reward}")
        print(f"Done Flag: {done}")

        if done:
            print(
                "Episode finished "
                "(Robot ran out of battery and needed rescue)."
            )
            break

    env.close()

else:
    print("\nEnvironment not created, skipping simulation.")

Successfully created RecyclingRobot-v0 environment.

Action Space: Discrete(3)
Observation Space: Discrete(2)

Initial State:
Current State: 0 (High Battery)

Simulation for a few steps:

--- Step 1 ---
Action Chosen: 1 (Wait)
Current State: 0 (High Battery)
Reward Received: 0.5
Done Flag: False

--- Step 2 ---
Action Chosen: 2 (Recharge)
Current State: 0 (High Battery)
Reward Received: 0.0
Done Flag: False

--- Step 3 ---
Action Chosen: 1 (Wait)
Current State: 0 (High Battery)
Reward Received: 0.5
Done Flag: False

--- Step 4 ---
Action Chosen: 0 (Search)
Current State: 0 (High Battery)
Reward Received: 2.0
Done Flag: False

--- Step 5 ---
Action Chosen: 0 (Search)
Current State: 0 (High Battery)
Reward Received: 2.0
Done Flag: False

--- Step 6 ---
Action Chosen: 0 (Search)
Current State: 1 (Low Battery)
Reward Received: 2.0
Done Flag: False

--- Step 7 ---
Action Chosen: 0 (Search)
Current State: 0 (High Battery)
Reward Received: -3.0
Done Flag: True
Episode finished (Robot ran out 